# Notebook 3: ESM2 + LoRA Fine-Tuning

**Prerequisite:** run `python run_preprocessing.py` first.

Fine-tunes ESM2 with LoRA adapters for multi-label GO term prediction.
After training, the model can be pushed to HuggingFace Hub.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
import torch.nn as nn
import yaml
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import DataLoader

from models.esm2_classifier import build_esm2_classifier, ESM2Classifier
from dataset import ESM2Dataset, collate_fn_pad
from evaluate import compute_fmax, compute_aupr, compute_coverage, load_ia_weights
from preprocessing import load_processed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

OBO_PATH = '../data/raw/Train/go-basic.obo'
IA_PATH  = '../data/raw/IA.tsv'

## 1. Configuration

In [ ]:
with open('../config/esm2_lora.yaml') as f:
    cfg = yaml.safe_load(f)

# Use the small 8M model for fast experimentation; swap to esm2_t33_650M_UR50D for best results
cfg['model']['base_model'] = 'facebook/esm2_t6_8M_UR50D'
cfg['model']['max_length']  = 512
cfg['training']['num_epochs']   = 10
cfg['training']['batch_size']   = 8
cfg['training']['gradient_accumulation_steps'] = 2

ONTOLOGY   = 'BPO'
DATA_PATH  = f'../data/processed/cafa6_{ONTOLOGY.lower()}.pkl'
OUTPUT_DIR = f'../outputs/esm2_lora_{ONTOLOGY.lower()}'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Training on ontology: {ONTOLOGY}')
print(f'Base model: {cfg["model"]["base_model"]}')

## 2. Load Data

In [ ]:
data      = load_processed(DATA_PATH)
go_terms  = data['go_terms']
num_labels = len(go_terms)
ia_weights = load_ia_weights(IA_PATH, go_terms)
print(f'GO terms ({ONTOLOGY}): {num_labels:,}')
print(f'Train: {len(data["train"]["protein_ids"]):,} | Val: {len(data["val"]["protein_ids"]):,} | Test: {len(data["test"]["protein_ids"]):,}')

tokenizer = AutoTokenizer.from_pretrained(cfg['model']['base_model'])
max_len   = cfg['model']['max_length']

def make_loader(split_key, shuffle, bs=None):
    split = data[split_key]
    bs = bs or cfg['training']['batch_size']
    ds = ESM2Dataset(split['sequences'], tokenizer, max_len, split['labels'], split['protein_ids'])
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      collate_fn=collate_fn_pad, num_workers=0, pin_memory=(DEVICE.type=='cuda'))

train_loader = make_loader('train', shuffle=True)
val_loader   = make_loader('val',   shuffle=False, bs=cfg['training']['batch_size']*2)
test_loader  = make_loader('test',  shuffle=False, bs=cfg['training']['batch_size']*2)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 3. Build ESM2 + LoRA Model

In [ ]:
model = build_esm2_classifier(cfg, num_labels, go_terms).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 4. Training Loop

In [ ]:
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg['training']['learning_rate'],
    weight_decay=cfg['training']['weight_decay'],
)

num_epochs = cfg['training']['num_epochs']
grad_accum = cfg['training']['gradient_accumulation_steps']
total_steps  = (len(train_loader) // grad_accum) * num_epochs
warmup_steps = int(total_steps * cfg['training']['warmup_ratio'])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

use_fp16 = cfg['training'].get('fp16', False) and DEVICE.type == 'cuda'
scaler   = torch.cuda.amp.GradScaler() if use_fp16 else None

history = {'train_loss': [], 'val_loss': [], 'fmax': [], 'aupr': []}

@torch.no_grad()
def eval_loader(loader, mdl=None):
    mdl = mdl or model
    mdl.eval()
    logits_l, labels_l = [], []
    total_loss = 0.0
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labs = batch['labels'].to(DEVICE)
        out  = mdl(ids, mask, labs)
        total_loss += out['loss'].item()
        logits_l.append(out['logits'].cpu().float().numpy())
        labels_l.append(labs.cpu().numpy())
    scores = torch.sigmoid(torch.tensor(np.concatenate(logits_l))).numpy()
    y_true = np.concatenate(labels_l)
    return y_true, scores, total_loss / len(loader)

best_fmax = 0.0
best_ckpt = f'{OUTPUT_DIR}/best'

for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}/{num_epochs}')):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labs = batch['labels'].to(DEVICE)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_fp16):
            out  = model(ids, mask, labs)
            loss = out['loss'] / grad_accum

        (scaler.scale(loss) if scaler else loss).backward()
        epoch_loss += loss.item() * grad_accum

        if (step + 1) % grad_accum == 0:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    y_true, scores, val_loss = eval_loader(val_loader)
    fmax, best_t = compute_fmax(y_true, scores)
    aupr = compute_aupr(y_true, scores)
    cov  = compute_coverage(scores, best_t)

    tl = epoch_loss / len(train_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(val_loss)
    history['fmax'].append(fmax)
    history['aupr'].append(aupr)

    print(f'Epoch {epoch:02d} | train={tl:.4f} | val={val_loss:.4f} | '
          f'Fmax={fmax:.4f} (t={best_t:.2f}) | AUPR={aupr:.4f} | Coverage={cov:.3f}')

    if fmax > best_fmax:
        best_fmax = fmax
        model.save_pretrained(best_ckpt)
        print(f'  >>> Best Fmax={fmax:.4f} — checkpoint saved')

## 5. Training Curves

In [ ]:
epochs = list(range(1, num_epochs + 1))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], 'o-', label='Train')
axes[0].plot(epochs, history['val_loss'],   's-', label='Val')
axes[0].set(xlabel='Epoch', ylabel='BCE Loss', title='Loss')
axes[0].legend()

axes[1].plot(epochs, history['fmax'], 'o-', color='green')
axes[1].set(xlabel='Epoch', ylabel='Fmax', title=f'Validation Fmax ({ONTOLOGY})')

axes[2].plot(epochs, history['aupr'], 'o-', color='purple')
axes[2].set(xlabel='Epoch', ylabel='AUPR', title=f'Validation AUPR ({ONTOLOGY})')

plt.tight_layout()
plt.savefig(f'../reports/esm2_lora_{ONTOLOGY.lower()}_training.png', dpi=150)
plt.show()
print(f'Best Fmax: {best_fmax:.4f}')

## 6. Test Set Evaluation

In [ ]:
from evaluate import compute_smin, compute_precision_recall_curve

best_model = ESM2Classifier.from_pretrained(best_ckpt).to(DEVICE)
y_true_ts, scores_ts, _ = eval_loader(test_loader, best_model)

fmax_ts, t_ts = compute_fmax(y_true_ts, scores_ts)
aupr_ts  = compute_aupr(y_true_ts, scores_ts)
cov_ts   = compute_coverage(scores_ts, t_ts)
smin_ts, _ = compute_smin(y_true_ts, scores_ts, ia_weights)

print(f'=== Test Set Results ({ONTOLOGY}) ===')
print(f'Fmax     : {fmax_ts:.4f}  (threshold={t_ts:.2f})')
print(f'AUPR     : {aupr_ts:.4f}')
print(f'Smin     : {smin_ts:.4f}')
print(f'Coverage : {cov_ts:.4f}')

prec, rec, thresholds = compute_precision_recall_curve(y_true_ts, scores_ts)
plt.figure(figsize=(6, 5))
plt.plot(rec, prec, lw=2)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title(f'ESM2 LoRA — {ONTOLOGY} | Fmax={fmax_ts:.3f}, AUPR={aupr_ts:.3f}')
plt.tight_layout()
plt.savefig(f'../reports/esm2_{ONTOLOGY.lower()}_pr_curve.png', dpi=150)
plt.show()

## 7. Push to HuggingFace Hub

In [ ]:
HUB_REPO = 'your-username/cafa6-esm2-lora-bpo'   # <-- change this
HF_TOKEN = 'hf_...'                                # <-- or run: huggingface-cli login

# Option A: push LoRA adapters only (~50 MB, base model loaded from Hub at inference)
# best_model.push_to_hub(HUB_REPO, token=HF_TOKEN)

# Option B: merge LoRA into base weights, push full standalone model
# best_model.merge_and_push(HUB_REPO, token=HF_TOKEN)

print('Uncomment one option above and set your HuggingFace credentials.')